<a href="https://colab.research.google.com/github/Ars160/DeepLearning/blob/main/Assignment06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

file_url = 'https://raw.githubusercontent.com/ardakshalkar/NLP2020_Assignment_01/main/data/StateGrants.csv'
df = pd.read_csv(file_url)

print("Первые строки данных:")
print(df.head())

names = df['РАХМЕТУЛЛА АСАНӘЛІ'].tolist()

# Шаг 2: Подготовка символов и словарей
kazakh_chars = "аәбвгғдеёжзийклмнңоөпрстуұүфхһцчшщъыіьэюя SF"
char2idx = {ch: i for i, ch in enumerate(kazakh_chars)}
idx2char = {i: ch for ch, i in char2idx.items()}

# Шаг 3: Подготовка данных для обучения
# Добавляем 'S' в начало и 'F' в конец каждого имени
processed_names = ['S' + str(name).lower() + 'F' for name in names if isinstance(name, str)]

print("Первые обработанные имена:")
print(processed_names[:5])

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)  # Слой эмбеддингов
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)  # RNN слой
        self.fc = nn.Linear(hidden_size, vocab_size)  # Полносвязный слой

    def forward(self, x):
        x = self.embedding(x)  # Преобразуем индексы в эмбеддинги
        out, _ = self.rnn(x)   # Проходим через RNN
        out = self.fc(out)     # Преобразуем в вероятности символов
        return out

# Шаг 5: Инициализация модели, функции потерь и оптимизатора
vocab_size = len(kazakh_chars)  # Размер словаря (42)
embed_size = 10                 # Размер эмбеддинга
hidden_size = 50                # Размер скрытого слоя
model = CharRNN(vocab_size, embed_size, hidden_size)
criterion = nn.CrossEntropyLoss()  # Функция потерь
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Оптимизатор

# Шаг 6: Обучение модели
num_epochs = 10
for epoch in range(num_epochs):
    for name in processed_names:

        if not all(c in kazakh_chars for c in name):
            continue

        # Вход: первые N-1 символов
        input_seq = [char2idx[c] for c in name[:-1]]
        # Цель: символы с 1 до N
        target_seq = [char2idx[c] for c in name[1:]]

        # Преобразуем в тензоры
        x = torch.tensor([input_seq], dtype=torch.long)
        y = torch.tensor(target_seq, dtype=torch.long)

        # Прямой проход
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out.view(-1, vocab_size), y)

        # Обратный проход и обновление весов
        loss.backward()
        optimizer.step()

    print(f"Эпоха {epoch+1}, Потеря: {loss.item()}")

# Шаг 7: Функция для генерации имени
def generateName(startText, model, char2idx, idx2char, max_len=30):
    model.eval()  # Переводим модель в режим оценки
    sequence = [char2idx['S']] + [char2idx[c] for c in startText.lower()]  # Начальная последовательность

    while len(sequence) < max_len:
        x = torch.tensor([sequence], dtype=torch.long)
        out = model(x)
        # Исправляем: берем индекс последнего символа из первого элемента батча
        next_char_idx = out[0, -1].argmax().item()  # [0, -1] выбирает последний символ из первого батча
        sequence.append(next_char_idx)
        if idx2char[next_char_idx] == 'F':  # Останавливаемся, если достигнут конец
            break

    return ''.join([idx2char[idx] for idx in sequence])  # Преобразуем в строку

# Шаг 8: Генерация имени и вывод результата
start_text = "ар"  # Начальный текст (можно изменить, например, на "ай")
generated_name = generateName(start_text, model, char2idx, idx2char)
print(f"Сгенерированное имя: {generated_name}")

Первые строки данных:
                РАХМЕТУЛЛА АСАНӘЛІ
0           Жамашев Ержан Жеңісұлы
1       ЗАМАНБЕК МУХАММЕД ЖЕҢІСҰЛЫ
2         ЖҰМАЖАНОВ МИРАС МАРАТҰЛЫ
3      Тлеулесов Максут Кайратович
4  ДҮЙСЕНБЕК НҰРДӘУЛЕТ ЖАҚСЫЛЫҚҰЛЫ
Первые обработанные имена:
['Sжамашев ержан жеңісұлыF', 'Sзаманбек мухаммед жеңісұлыF', 'Sжұмажанов мирас маратұлыF', 'Sтлеулесов максут кайратовичF', 'Sдүйсенбек нұрдәулет жақсылықұлыF']
Эпоха 1, Потеря: 2.5010757446289062
Эпоха 2, Потеря: 2.417924642562866
Эпоха 3, Потеря: 2.3681490421295166
Эпоха 4, Потеря: 2.304025650024414
Эпоха 5, Потеря: 2.244464635848999
Эпоха 6, Потеря: 2.2063586711883545
Эпоха 7, Потеря: 2.1845855712890625
Эпоха 8, Потеря: 2.1687440872192383
Эпоха 9, Потеря: 2.151228427886963
Эпоха 10, Потеря: 2.131718873977661
Сгенерированное имя: Sарман александрова александро
